In [4]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [5]:
X,y = make_classification(n_samples=1000,
                          n_features=10,
                          n_redundant=8,
                          weights=[0.9,0.1],
                          flip_y=0,
                          random_state=42)
np.unique(y,return_counts=True)

(array([0, 1]), array([900, 100]))

In [6]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,stratify=y,random_state=42)

#### Experiment 1: Train Logistic Regression

In [7]:
params={
    'solver':'lbfgs',
    'max_iter':1000,
    'multi_class':'auto',
    'random_state':8888
}
lr=LogisticRegression(**params)
lr.fit(X_train,y_train)
y_pred=lr.predict(X_test)
report=classification_report(y_test,y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [8]:
report_dict=classification_report(y_test,y_pred,output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [9]:
import mlflow

In [10]:
mlflow.set_experiment('First Experiment')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy':report_dict['accuracy'],
        'recall_class_0':report_dict['0']['recall'],
        'recall_class_1':report_dict['1']['recall'],
        'f1_score_class_0':report_dict['0']['f1-score'],
        'f1_score_class_1':report_dict['1']['f1-score'],
        'f1_score_macro_avg':report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(lr,'Logistic Regression')

2025/11/27 17:27:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/27 17:28:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run worried-snail-994 at: http://127.0.0.1:5000/#/experiments/192307785387825820/runs/ed4a476e2cf64de99ac0def2b0915152
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/192307785387825820


#### Experiment 2: Train Random Forest Classifier

In [ ]:
rf_clf =RandomForest

#### Experiment 3: Train XGBoost

In [12]:
xgb_clf=XGBClassifier(use_label_encoder=False,eval_metrics='logloss')
xgb_clf.fit(X_train,y_train)
y_pred_xgb=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### Experiment 4: Handle class Imbalance using SMOTETomek and then train XGBoost

In [14]:
from imblearn.combine import SMOTETomek

sat= SMOTETomek(random_state=42)
X_train_res,y_train_res = sat.fit_resample(X_train,y_train)
np.unique(y_train_res,return_counts=True)

(array([0, 1]), array([619, 619]))

In [15]:
xgb_clf.fit(X_train_res,y_train_res)
y_pred_xgb_smote=xgb_clf.predict(X_test)
print(classification_report(y_test,y_pred_xgb_smote))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       270
           1       0.81      0.83      0.82        30

    accuracy                           0.96       300
   macro avg       0.89      0.91      0.90       300
weighted avg       0.96      0.96      0.96       300



In [16]:
models=[
    (
      'LogisticRegression',
        LogisticRegression(C=1,solver='liblinear'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'Random Forest',
        RandomForestClassifier(n_estimators=30,max_depth=3),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train,y_train),
        (X_test,y_test)
    ),
    (
        'XGBClassifier with SMOTE',
        XGBClassifier(use_label_encoder=False,eval_metrics='logloss'),
        (X_train_res,y_train_res),
        (X_test,y_test)
    )
]

In [17]:
reports= []
for model_name,model,train_set,test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test,y_pred,output_dict=True)
    reports.append(report)

In [18]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669725,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.91326105087573,
   'support': 300.0}},
 {'0': {'precision': 0.9676258992805755,
   'recall': 0.9962962962962963,
   'f1-score': 0.9817518248175182,
   'support': 270.0},
  '1': {'precision': 0.9545454545454546,
   'recall': 0.7,
   'f1-score': 0.8076923076923077,
   'support': 30.0},
  'accuracy': 0.9666666666666667,
  'macro avg': {'precision': 0.961085676913015,
   'recall': 0.8481481481481481,
   'f1-score': 0.8947220662549129,
   'support': 300.0},
  'weighted avg': {'precision': 0.9663

In [23]:
mlflow.set_experiment('Anomaly_detection')
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

for i,element in enumerate(models):
    model_name=element[0]
    model=element[1]
    report=reports[i]
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name',model_name)
        
        mlflow.log_metrics({
        'accuracy':report['accuracy'],
        'recall_class_0':report['0']['recall'],
        'recall_class_1':report['1']['recall'],
        'f1_score_class_0':report['0']['f1-score'],
        'f1_score_class_1':report['1']['f1-score'],
        'f1_score_macro_avg':report['macro avg']['f1-score']
         })
        if 'XGB' in model_name:
             mlflow.xgboost.log_model(model,f'{model_name}')
        else:
             mlflow.sklearn.log_model(model,f'{model_name}')


2025/11/27 17:41:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/27 17:41:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/76056649f7054a3aa0ee7d4fbe4b3e85
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/27 17:41:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/27 17:41:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/27 17:41:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/39a25f1aae4b4f2fbb4805767a96cab4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/27 17:41:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/27 17:41:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/719f5cb92b8a4caca3ae6d7cb65b18e0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


2025/11/27 17:41:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run XGBClassifier with SMOTE at: http://127.0.0.1:5000/#/experiments/801989169201354244/runs/415a692f9e1c462c8e71b5dd080788e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/801989169201354244


#### Register the model

In [26]:
model_name = 'LogisticRegression'
run_id = input('Enter Run ID')
model_uri=f'runs:/{run_id}/{model_name}'
result = mlflow.register_model(
     model_uri,model_name
)

Enter Run ID 76056649f7054a3aa0ee7d4fbe4b3e85


Successfully registered model 'LogisticRegression'.
2025/11/27 17:52:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 76056649f7054a3aa0ee7d4fbe4b3e85 has no artifacts at artifact path 'LogisticRegression', registering model based on models:/m-a70a8d8a2a5b4cb284a943d912044dbb instead
2025/11/27 17:52:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LogisticRegression, version 1
Created version '1' of model 'LogisticRegression'.


#### Load the model

In [33]:
model_version = 1
model_uri = f'models:/{model_name}@challenger'
loaded_model = mlflow.sklearn.load_model(model_uri)
y_pred=loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [34]:
result

<ModelVersion: aliases=[], creation_timestamp=1764262351846, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1764262351846, metrics=None, model_id=None, name='LogisticRegression', params=None, run_id='76056649f7054a3aa0ee7d4fbe4b3e85', run_link='', source='models:/m-a70a8d8a2a5b4cb284a943d912044dbb', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [36]:
dev_model_uri = f'models:/{model_name}@challenger'
prod_model='anomaly-detection-prod'
client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri,dst_name=prod_model)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'LogisticRegression' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1764265533830, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1764265533830, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='76056649f7054a3aa0ee7d4fbe4b3e85', run_link='', source='models:/LogisticRegression/1', status='READY', status_message=None, tags={}, user_id='', version='1'>